# 04 — Random Forest Regression

Random Forest is trained on the leakage-controlled feature set. The fitted **entire pipeline** is saved so preprocessing and the estimator stay together.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, joblib, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
PROJECT_ROOT=Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent; DATA_PATH=PROJECT_ROOT/"data"/"Crop Yield Data.csv"; MODEL_DIR=PROJECT_ROOT/"models"; OUTPUT_DIR=PROJECT_ROOT/"outputs"; RANDOM_STATE=42
df=pd.read_csv(DATA_PATH).dropna(subset=["Yield"]).reset_index(drop=True)
X=df.drop(columns=["Yield","Production"]).copy(); y=df["Yield"].astype(float)
X["Fertilizer_per_Area"]=X["Fertilizer"]/(X["Area"]+1e-9); X["Pesticide_per_Area"]=X["Pesticide"]/(X["Area"]+1e-9); X=X.replace([np.inf,-np.inf],np.nan)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,random_state=RANDOM_STATE)
numeric_features=X.select_dtypes(include=np.number).columns.tolist(); categorical_features=X.select_dtypes(exclude=np.number).columns.tolist()
tree_preprocessor=ColumnTransformer([("numeric",SimpleImputer(strategy="median"),numeric_features),("categorical",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("encoder",OrdinalEncoder(handle_unknown="use_encoded_value",unknown_value=-1))]),categorical_features)])

In [ ]:
from sklearn.ensemble import RandomForestRegressor
rf=Pipeline([("preprocessor",tree_preprocessor),("model",RandomForestRegressor(n_estimators=50,max_features=.8,n_jobs=-1,random_state=42))])
rf.fit(X_train,y_train); predictions=rf.predict(X_test)
mae=mean_absolute_error(y_test,predictions); mse=mean_squared_error(y_test,predictions); rmse=np.sqrt(mse); r2=r2_score(y_test,predictions)
print(f"MAE : {mae:.4f}\nMSE : {mse:.4f}\nRMSE: {rmse:.4f}\nR²  : {r2:.4f}")
joblib.dump(rf,MODEL_DIR/"random_forest_pipeline.pkl",compress=3)

Random Forest test evaluation
MAE : 11.811504
MSE : 37551.493416
RMSE: 193.782077
R²  : 0.946854
Model saved to models/random_forest_pipeline.pkl


In [ ]:
res=pd.DataFrame({"Actual":y_test.values,"Predicted":predictions}); display(res.head(10))
plt.figure(figsize=(7,6)); plt.scatter(y_test,predictions,alpha=.35); lim=max(y_test.max(),predictions.max()); plt.plot([0,lim],[0,lim],"--"); plt.xlabel("Actual Yield"); plt.ylabel("Predicted Yield"); plt.title("Random Forest — Actual vs Predicted"); plt.tight_layout(); plt.savefig(OUTPUT_DIR/"rf_actual_vs_predicted.png",dpi=160); plt.show()

First 10 predictions are available in the generated result table; actual-vs-predicted plot saved to outputs/rf_actual_vs_predicted.png


**Interpretation:** Random Forest provides a strong nonlinear baseline. Its test metrics are recorded later in the unified comparison table; the test set remains untouched by hyperparameter tuning.